# SIP Dropout Project: Dataset Discovery

**Milestone 1:** validate a public dataset and decide what it can and cannot tell us.

## Important scope note
This notebook uses the IBM Telco Customer Churn sample as a **workflow-learning proxy**, not as SIP data. It contains telecom customers and a historical churn label, not mutual-fund SIP instalments, mandate bounces, or SIP cancellation reasons.

We will also use public AMFI SIP statistics for domain context. Those statistics are aggregated by month, so they cannot by themselves train an individual-SIP risk model.

### Sources
- IBM Telco sample CSV: https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv
- IBM sample documentation: https://www.ibm.com/docs/en/cognos-analytics/12.0.x?topic=samples-telco-customer-churn
- AMFI mutual-fund / SIP statistics: https://www.amfiindia.com/mutual-fund
- Publicly listed AMFI monthly dataset: https://dataful.in/datasets/20840/

**Data provenance rule:** do not describe proxy or simulated data as real SIP investor data.

## 1. Load the libraries

We use `pandas` to work with tabular data. `matplotlib` is for basic plots. If a library is missing in your Jupyter environment, install it in a separate notebook cell using `%pip install pandas matplotlib`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

## 2. Load the public proxy dataset

This reads a CSV directly from its public raw-file URL. An internet connection is required.

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Rows and columns:", df.shape)
display(df.head())

## 3. Inspect the schema

We need to know what each row represents and what columns exist before making modelling decisions.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

## 4. Inspect the target label

`Churn` describes the telecom outcome in this dataset. It is **not** our final SIP target. We are checking how a labelled classification dataset is structured.

In [ ]:
print(df["Churn"].value_counts(dropna=False))
print("\nProportions:")
print(df["Churn"].value_counts(normalize=True, dropna=False).round(3))

## 5. Check data quality

Look for missing values and duplicate rows. These checks do not automatically mean we should drop anything; they tell us what to investigate.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())

## 6. Quick visual inspection

A simple chart helps us see how common each label is. This is an exploratory view, not a model evaluation.

In [ ]:
label_counts = df["Churn"].value_counts()

ax = label_counts.plot(kind="bar")
ax.set_title("Telecom churn label counts (proxy dataset)")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. SIP project data-gap assessment

Use this checklist after running the notebook.

| Needed for our SIP project | In the Telco proxy? | Why it matters |
|---|---|---|
| SIP instalment history | No | Needed to identify missed or bounced instalments |
| SIP cancellation/discontinuation date | No, only telecom churn label | Needed to define a SIP lapse outcome |
| Investor/SIP identifier over time | No SIP identifiers | Needed for longitudinal SIP-level prediction |
| SIP amount and planned frequency | No SIP-specific values | Needed for commitment and payment-pattern features |
| Market drawdown / fund performance | No | Must be joined from appropriate market/fund data |
| Stated reason for discontinuation | No SIP reason label | Needed to train a reason classifier |
| General retention features | Partly analogous | Useful only for workflow practice, not direct transfer |

## 8. Decision

After running the cells, record:
1. Does the CSV load?
2. What is its shape?
3. What is the churn label distribution?
4. Which columns have missing values?
5. What prevents this dataset from directly training a SIP dropout model?

**Current decision:** use this dataset to practise the data workflow, while searching for SIP-specific individual-level data. Use AMFI aggregate statistics for context and trend analysis, not as if each monthly row were an individual SIP.